# Projecting Onto a Basis That Isn't One

**Topic:** Projection and least squares &nbsp;·&nbsp; **Difficulty:** Medium

A colleague hands you five "independent" reference signals and asks you to
project the measured signal onto the space they span. The signals are not
independent: one of them is an exact linear combination of two others, which
means the normal equations you were about to write down involve a singular
matrix.

The projection itself is still perfectly well defined. Only your method for
computing it is broken.

## The data (created in the next cell)

- `A` `(300, 5)`: five reference signals as columns. **Rank 4**, not 5.
- `b` `(300,)`: the measured signal.

## Your task

With **NumPy only**:

1. Compute `p`, the orthogonal projection of `b` onto the column space of `A`.
   `np.linalg.inv(A.T @ A)` will not work, because the matrix is singular. Use
   `np.linalg.pinv` or an orthonormal basis from `np.linalg.svd`, keeping only
   the singular vectors whose singular value is not negligible. A plain
   `np.linalg.qr(A)` returns five orthonormal columns here, one more than the
   column space has, so `Q @ Q.T @ b` is not the projection.
2. Compute the residual `r = b - p`.
3. Set `answers` to a dict with `"p"` `(300,)`, `"residual_norm"` (float), and
   `"rank"` (int, the rank of `A`).

## Evaluation

The projection is compared to 1e-6, and the evaluation cell separately checks
that your residual is actually orthogonal to every column of `A`.

In [ ]:
import numpy as np

rng = np.random.default_rng(31)
t = np.linspace(0, 4 * np.pi, 300)
a0, a1, a2 = np.sin(t), np.cos(t), np.sin(2 * t)
a3 = 0.5 * a0 - 1.5 * a1          # exactly dependent on a0 and a1
a4 = np.cos(3 * t)
A = np.column_stack([a0, a1, a2, a3, a4])
b = 2.0 * a0 - 0.7 * a2 + 1.1 * a4 + rng.normal(0, 0.4, 300)
print(f"A {A.shape}, numerical rank = {np.linalg.matrix_rank(A)}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the data created above to solve the task, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("p", [300], 1.106463157774698, -14.691749486220257, 1e-06),
    ("residual_norm", [], 6.304983621850059, -1.0556939410805046, 1e-06),
    ("rank", [], 4.0, -0.6697520592579966, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "p" in answers:
    _r = np.asarray(b, dtype=float) - np.asarray(answers["p"], dtype=float)
    _orth = float(np.abs(A.T @ _r).max())
    print(f"  {'residual _|_ columns':<24} {'ok' if _orth < 1e-8 else 'wrong'}  (max |A.T r| = {_orth:.2e})")
    _ok = _ok and _orth < 1e-8

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The pseudo-inverse gives the minimum-norm least-squares coefficients, and
`A @ pinv(A) @ b` is the projection regardless of rank. Equivalently, take a
thin SVD, keep the singular vectors with non-negligible singular values, and sum
the components of `b` along them.

```python
import numpy as np

p = A @ (np.linalg.pinv(A) @ b)
r = b - p

answers = {
    "p": p,
    "residual_norm": float(np.linalg.norm(r)),
    "rank": int(np.linalg.matrix_rank(A)),
}
```